# FinanceRAG — Notebook d'Exploration

**Objectif** : Tester le pipeline RAG de bout en bout, évaluer la qualité du retrieval et des réponses.

---
**Sections :**
1. Setup & Configuration
2. Chargement et inspection des documents
3. Découpage en chunks (analyse)
4. Test du retrieval (pertinence)
5. Test du pipeline RAG complet
6. Évaluation qualitative

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# FinanceRAG
from src.config import CHUNK_SIZE, CHUNK_OVERLAP, EMBEDDING_MODEL
from src.document_loader import load_directory, get_document_stats
from src.indexing import split_documents, build_vectorstore
from src.retriever import FinanceRetriever

print('✓ Imports OK')
print(f'  Chunk size: {CHUNK_SIZE} | Overlap: {CHUNK_OVERLAP}')
print(f'  Embedding: {EMBEDDING_MODEL}')

2026-10-18 23:19:23.941 | WARNING  | src.config:<module>:28 - ANTHROPIC_API_KEY non définie — les appels LLM échoueront.
2026-10-18 23:19:23.942 | INFO     | src.config:<module>:73 - Config chargée — modèle : claude-3-5-sonnet-20241022 | chunks : 800/100


✓ Imports OK
  Chunk size: 800 | Overlap: 100
  Embedding: sentence-transformers/all-MiniLM-L6-v2


## 1. Chargement des documents

In [ ]:
# Charger tous les documents du répertoire data/raw
docs = load_directory('../data/raw')
stats = get_document_stats(docs)

print(f"📄 Documents chargés :")
print(f"  - Fichiers sources  : {stats['nb_documents']}")
print(f"  - Segments totaux   : {stats['nb_segments']}")
print(f"  - Caractères moyens : {stats['avg_chars']:,}")
print(f"  - Caractères totaux : {stats['total_chars']:,}")
print()
print('Sources :')
for s in stats['sources']:
    print(f'  • {s}')

In [ ]:
# Inspecter un document
if docs:
    doc = docs[0]
    print(f"Source : {doc.metadata['source']}")
    print(f"Page   : {doc.metadata.get('page', 'N/A')}")
    print(f"Chars  : {len(doc.page_content)}")
    print()
    print("Extrait (200 premiers caractères) :")
    print("-" * 50)
    print(doc.page_content[:200])

## 2. Analyse du Découpage (Chunking)

In [ ]:
# Découper les documents
chunks = split_documents(docs)

# Distribution des tailles de chunks
chunk_sizes = [len(c.page_content) for c in chunks]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distribution des Chunks', fontsize=14, fontweight='bold')

# Histogramme
axes[0].hist(chunk_sizes, bins=30, color='#1B2A4A', alpha=0.8, edgecolor='white')
axes[0].axvline(CHUNK_SIZE, color='#C9A84C', linewidth=2, label=f'Chunk size={CHUNK_SIZE}')
axes[0].set_xlabel('Taille (caractères)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution des tailles')
axes[0].legend()

# Statistiques
import numpy as np
stats_data = {
    'Métrique': ['Total chunks', 'Taille médiane', 'Taille moyenne', 'Taille min', 'Taille max'],
    'Valeur': [len(chunks), int(np.median(chunk_sizes)), int(np.mean(chunk_sizes)), 
               min(chunk_sizes), max(chunk_sizes)]
}
df_stats = pd.DataFrame(stats_data)
axes[1].axis('off')
table = axes[1].table(cellText=df_stats.values, colLabels=df_stats.columns, 
                      loc='center', cellLoc='left')
table.auto_set_font_size(True)

plt.tight_layout()
plt.savefig('../docs/chunk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ {len(chunks)} chunks générés')

## 3. Test du Retrieval

In [ ]:
# Construire l'index (sans sauvegarder) pour tester
print('Construction de l\'index FAISS (peut prendre 1-2 min)...')
vectorstore = build_vectorstore(chunks, save_path='../vectorstore/faiss_index')
retriever = FinanceRetriever(vectorstore, top_k=3)
print('✓ Index construit')

In [ ]:
# Tester plusieurs requêtes
test_queries = [
    "Quels sont les frais de gestion ?",
    "Quelle est la garantie décès ?",
    "Comment calculer le taux d'intérêt ?",
    "Quels sont les critères d'éligibilité ?",
]

for query in test_queries:
    print(f"\n🔍 '{query}'")
    print("-" * 60)
    retrieved = retriever.retrieve(query)
    for i, doc in enumerate(retrieved, 1):
        score = doc.metadata.get('similarity_score', 'N/A')
        source = doc.metadata.get('source', '?')
        preview = doc.page_content[:100].replace('\n', ' ')
        print(f"  [{i}] {source} | score={score}")
        print(f"      {preview}...")

## 4. Pipeline RAG Complet

In [ ]:
# ⚠️ Nécessite ANTHROPIC_API_KEY dans .env
from src.rag_pipeline import FinanceRAGPipeline

pipeline = FinanceRAGPipeline()

question = "Quels sont les frais de gestion annuels de l'assurance vie ?"
response = pipeline.query(question)

print(f"Question : {question}")
print("=" * 60)
print(f"Réponse :\n{response.answer}")
print("=" * 60)
print(f"Sources : {', '.join(response.sources)}")
print(f"Tokens  : {response.total_tokens} (in={response.input_tokens}, out={response.output_tokens})")

## 5. Évaluation qualitative

| Critère | Description | Score (1-5) |
|---------|-------------|-------------|
| Pertinence du retrieval | Les chunks récupérés sont-ils pertinents ? | — |
| Fidélité | La réponse est-elle fidèle aux sources ? | — |
| Citation des sources | Les sources sont-elles bien citées ? | — |
| Refus hors-sujet | Répond-il "Je ne sais pas" quand il le faut ? | — |
| Qualité du français | La réponse est-elle fluide et précise ? | — |

In [ ]:
# Test de refus : question hors-sujet
response_hors_sujet = pipeline.query("Quel est le temps qu'il fait à Paris aujourd'hui ?")
print("Question hors-sujet :")
print(response_hors_sujet.answer)
# Attendu : message indiquant que l'information n'est pas dans les documents